In [1]:
# Cellule 1 — imports
import pandas as pd
import numpy as np
import geopandas as gpd
import requests
import time
import os

os.makedirs("../data/raw/climat_nasa_communal", exist_ok=True)
os.makedirs("../data/raw/limites", exist_ok=True)
os.makedirs("../data/raw/pedologie", exist_ok=True)

print("Environnement prêt.")
print(f"Geopandas : {gpd.__version__}")

Environnement prêt.
Geopandas : 1.1.3


In [2]:
# Cellule 2 — extraction des rendements communaux
df_mais = pd.read_excel(
    "../data/raw/rendement_communale.xlsx",
    sheet_name="Maïs",
    header=0
)

# Exclure Bénin et les départements, garder uniquement les communes
a_exclure = [
    'Bénin', 'Atakora', 'Alibori', 'Atlantique', 'Borgou',
    'Collines', 'Donga', 'Kouffo', 'Littoral', 'Mono',
    'Ouémé', 'Plateau', 'Zou'
]

df_communes = df_mais[
    (df_mais["Indicateurs"] == "Rendement en kg/ha") &
    (~df_mais["Zones"].isin(a_exclure))
].copy()

print(f"Nombre de communes : {df_communes['Zones'].nunique()}")
print(f"Shape : {df_communes.shape}")
print(df_communes["Zones"].unique())

Nombre de communes : 77
Shape : (77, 33)
<StringArray>
[      'Boukoumbé',           'Cobly',           'Kérou',         'Kouandé',
          'Matéri',      'Natitingou',        'Péhounco',       'Tanguiéta',
    'Toucountouna',       'Banikoara',        'Gogounou',           'Kandi',
        'Karimama',      'Malanville',         'Segbana',   'Abomey-Calavi',
          'Allada',        'Kpomassè',          'Ouidah',          'Sô-Ava',
           'Toffo',    'Tori-Bossito',              'Zè',       'Bembèrèkè',
          'Kalalé',          'N'Dali',           'Nikki',         'Parakou',
          'Pèrèrè',         'Sinendé',       'Tchaourou',           'Bantè',
    'Dassa-Zounmè',         'Glazoué',          'Ouèssè',         'Savalou',
            'Savè',         'Bassila',         'Copargo',         'Djougou',
           'Ouaké',        'Aplahoué',      'Djakotomey',           'Dogbo',
      'Klouékanmè',            'Lalo',        'Toviklin',         'Cotonou',
         'Athiémé',  

In [3]:
# Cellule 3 — transformer en format long
cols_annees = [c for c in df_communes.columns if str(c).isdigit()]

df_long = df_communes.melt(
    id_vars=["Zones"],
    value_vars=cols_annees,
    var_name="annee",
    value_name="rendement_kg_ha"
)

df_long["annee"] = df_long["annee"].astype(int)
df_long = df_long.rename(columns={"Zones": "commune"})

# Filtrer 2003-2022
df_long = df_long[
    (df_long["annee"] >= 2003) &
    (df_long["annee"] <= 2022)
].copy()

# Convertir en t/ha
df_long["rendement_t_ha"] = df_long["rendement_kg_ha"] / 1000

# Supprimer les valeurs manquantes
print(f"Avant suppression NaN : {df_long.shape}")
df_long = df_long.dropna(subset=["rendement_t_ha"]).reset_index(drop=True)
print(f"Après suppression NaN : {df_long.shape}")
print(f"Communes : {df_long['commune'].nunique()}")
print(f"Années : {sorted(df_long['annee'].unique())}")
print(df_long.head(5))

Avant suppression NaN : (1540, 4)
Après suppression NaN : (1515, 4)
Communes : 77
Années : [np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
     commune  annee  rendement_kg_ha  rendement_t_ha
0  Boukoumbé   2003      1019.387755        1.019388
1      Cobly   2003      1622.168840        1.622169
2      Kérou   2003      1770.690480        1.770690
3    Kouandé   2003      1343.696682        1.343697
4     Matéri   2003       990.806223        0.990806


In [6]:
# Cellule 4 — coordonnées GPS centrales des communes
communes_shp = gpd.read_file("../data/raw/limites/Lim_IGN_Communes.shp")
communes_shp = communes_shp.set_crs(epsg=32631)

# Convertir en WGS84 pour avoir lat/lon
communes_wgs84 = communes_shp.to_crs(epsg=4326)
communes_wgs84["latitude"]  = communes_wgs84.geometry.centroid.y
communes_wgs84["longitude"] = communes_wgs84.geometry.centroid.x

coords = communes_wgs84[["NAME", "dep", "latitude", "longitude"]].copy()
coords.columns = ["commune", "departement", "latitude", "longitude"]

print(f"Communes avec coordonnées : {len(coords)}")
print(coords.head(10).to_string())

# Sauvegarder
coords.to_csv("../data/metadata/coordonnees_communes.csv", index=False)
print("\nSauvegardé dans data/metadata/coordonnees_communes.csv")

Communes avec coordonnées : 77
         commune departement  latitude  longitude
0         Abomey         Zou  7.191636   1.977448
1  Abomey-Calavi  Atlantique  6.504770   2.323752
2     Adja-Ouere     Plateau  6.997113   2.595750
3         Adjara       Oueme  6.493022   2.677677
4       Adjohoun       Oueme  6.709607   2.501189
5   Agbangnizoun         Zou  7.096289   1.963643
6        Aguegue       Oueme  6.475930   2.531540
7         Allada  Atlantique  6.678939   2.119782
8       Aplahoue      Couffo  7.177165   1.692379
9        Athieme        Mono  6.543677   1.728730

Sauvegardé dans data/metadata/coordonnees_communes.csv


C:\Users\pc\AppData\Local\Temp\ipykernel_12048\2575435027.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  communes_wgs84["latitude"]  = communes_wgs84.geometry.centroid.y
C:\Users\pc\AppData\Local\Temp\ipykernel_12048\2575435027.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  communes_wgs84["longitude"] = communes_wgs84.geometry.centroid.x


In [7]:
# Cellule 4 corrigée — centroïde calculé avant conversion
communes_shp = gpd.read_file("../data/raw/limites/Lim_IGN_Communes.shp")
communes_shp = communes_shp.set_crs(epsg=32631)

# Calculer le centroïde en projection métrique (correct)
communes_shp["centroid"] = communes_shp.geometry.centroid

# Convertir les centroïdes en WGS84
centroids_wgs84 = communes_shp.set_geometry("centroid").to_crs(epsg=4326)
communes_shp["latitude"]  = centroids_wgs84.geometry.y
communes_shp["longitude"] = centroids_wgs84.geometry.x

coords = communes_shp[["NAME", "dep", "latitude", "longitude"]].copy()
coords.columns = ["commune", "departement", "latitude", "longitude"]

print(f"Communes : {len(coords)}")
print(coords.head(5).to_string())
coords.to_csv("../data/metadata/coordonnees_communes.csv", index=False)
print("\nSauvegardé sans warnings.")

Communes : 77
         commune departement  latitude  longitude
0         Abomey         Zou  7.191636   1.977449
1  Abomey-Calavi  Atlantique  6.504754   2.323741
2     Adja-Ouere     Plateau  6.997098   2.595757
3         Adjara       Oueme  6.493022   2.677677
4       Adjohoun       Oueme  6.709604   2.501188

Sauvegardé sans warnings.


In [8]:
# Cellule 5 — téléchargement NASA POWER pour toutes les communes
parametres = "PRECTOTCORR,T2M,T2M_MAX,T2M_MIN,RH2M,ALLSKY_SFC_SW_DWN"

def telecharger_nasa_power_commune(nom, lat, lon):
    url = "https://power.larc.nasa.gov/api/temporal/monthly/point"
    params = {
        "parameters": parametres,
        "community":  "AG",
        "longitude":  lon,
        "latitude":   lat,
        "start":      "2003",
        "end":        "2022",
        "format":     "CSV",
    }
    response = requests.get(url, params=params, timeout=60)
    if response.status_code == 200:
        nom_fichier = nom.replace("/", "-").replace(" ", "_").replace("'", "")
        chemin = f"../data/raw/climat_nasa_communal/{nom_fichier}.csv"
        with open(chemin, "w", encoding="utf-8") as f:
            f.write(response.text)
        return True
    else:
        print(f"  ERREUR {response.status_code} : {nom}")
        return False

# Télécharger pour toutes les communes
total = len(coords)
for i, row in coords.iterrows():
    print(f"[{i+1}/{total}] {row['commune']}...")
    succes = telecharger_nasa_power_commune(
        row["commune"], row["latitude"], row["longitude"]
    )
    if succes:
        print(f"  OK")
    time.sleep(1.5)

print("\nTéléchargement terminé !")

[1/77] Abomey...
  OK
[2/77] Abomey-Calavi...
  OK
[3/77] Adja-Ouere...
  OK
[4/77] Adjara...
  OK
[5/77] Adjohoun...
  OK
[6/77] Agbangnizoun...
  OK
[7/77] Aguegue...
  OK
[8/77] Allada...
  OK
[9/77] Aplahoue...
  OK
[10/77] Athieme...
  OK
[11/77] Banikoara...
  OK
[12/77] Bante...
  OK
[13/77] Bassila...
  OK
[14/77] Bembereke...
  OK
[15/77] Bohicon...
  OK
[16/77] Bonou...
  OK
[17/77] Bopa...
  OK
[18/77] Boukoumbe...
  OK
[19/77] Cobli...
  OK
[20/77] Come...
  OK
[21/77] Copargo...
  OK
[22/77] Cotonou...
  OK
[23/77] Cove...
  OK
[24/77] Dangbo...
  OK
[25/77] Dassa-Zoume...
  OK
[26/77] Djakotomey...
  OK
[27/77] Djougou...
  OK
[28/77] Glazoue...
  OK
[29/77] Gogounou...
  OK
[30/77] Grand-Popo...
  OK
[31/77] Houeyogbe...
  OK
[32/77] Ifangni...
  OK
[33/77] Kalale...
  OK
[34/77] Kandi...
  OK
[35/77] Kerou...
  OK
[36/77] Klouekanme...
  OK
[37/77] Kouande...
  OK
[38/77] Kpomasse...
  OK
[39/77] Lokossa...
  OK
[40/77] Malanville...
  OK
[41/77] Materi...
  OK
[42/77] 

In [9]:
# Cellule 6 — lecture et fusion des fichiers NASA POWER communaux
from io import StringIO

mois_map = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4,
    "MAY": 5, "JUN": 6, "JUL": 7, "AUG": 8,
    "SEP": 9, "OCT": 10, "NOV": 11, "DEC": 12
}

def lire_nasa_power_commune(nom, chemin):
    with open(chemin, "r", encoding="utf-8") as f:
        lignes = f.readlines()

    debut = 0
    for i, ligne in enumerate(lignes):
        if "-END HEADER-" in ligne:
            debut = i + 1
            break

    contenu = "".join(lignes[debut:])
    df = pd.read_csv(StringIO(contenu))

    mois = ["JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","NOV","DEC"]
    df_long = df[["PARAMETER","YEAR"] + mois].melt(
        id_vars=["PARAMETER","YEAR"],
        value_vars=mois,
        var_name="MOIS",
        value_name="VALEUR"
    )

    df_pivot = df_long.pivot_table(
        index=["YEAR","MOIS"],
        columns="PARAMETER",
        values="VALEUR"
    ).reset_index()

    df_pivot.columns.name = None
    df_pivot["commune"] = nom
    df_pivot["annee"] = df_pivot["YEAR"]
    return df_pivot

dossier = "../data/raw/climat_nasa_communal"
frames = []

for _, row in coords.iterrows():
    nom = row["commune"]
    nom_fichier = nom.replace("/", "-").replace(" ", "_").replace("'", "")
    chemin = f"{dossier}/{nom_fichier}.csv"
    if os.path.exists(chemin):
        df = lire_nasa_power_commune(nom, chemin)
        frames.append(df)
    else:
        print(f"MANQUANT : {nom}")

climat_mensuel_communal = pd.concat(frames, ignore_index=True)
print(f"Dataset climatique mensuel communal : {climat_mensuel_communal.shape}")
print(climat_mensuel_communal.head(3))

Dataset climatique mensuel communal : (18480, 10)
   YEAR MOIS  ALLSKY_SFC_SW_DWN  PRECTOTCORR   RH2M    T2M  T2M_MAX  T2M_MIN  \
0  2003  APR              19.23         4.01  81.68  27.61    36.23    22.99   
1  2003  AUG              15.34         4.52  87.44  24.54    30.17    19.77   
2  2003  DEC              18.55         0.55  75.95  25.20    31.77    16.64   

  commune  annee  
0  Abomey   2003  
1  Abomey   2003  
2  Abomey   2003  


In [10]:
# Cellule 7 — agrégation annuelle par commune
def agreger_par_annee_commune(df):
    resultats = []

    for (commune, annee), groupe in df.groupby(["commune", "annee"]):
        row = {"commune": commune, "annee": annee}

        for mois_abr, mois_num in mois_map.items():
            ligne_mois = groupe[groupe["MOIS"] == mois_abr]
            if len(ligne_mois) > 0:
                row[f"precip_{mois_num:02d}_mm"]    = ligne_mois["PRECTOTCORR"].values[0] * [31,28,31,30,31,30,31,31,30,31,30,31][mois_num-1]
                row[f"temp_moy_{mois_num:02d}_c"]   = ligne_mois["T2M"].values[0]
                row[f"temp_max_{mois_num:02d}_c"]   = ligne_mois["T2M_MAX"].values[0]
                row[f"temp_min_{mois_num:02d}_c"]   = ligne_mois["T2M_MIN"].values[0]
                row[f"humidite_{mois_num:02d}_pct"] = ligne_mois["RH2M"].values[0]
                row[f"rayonnement_{mois_num:02d}"]  = ligne_mois["ALLSKY_SFC_SW_DWN"].values[0]

        # Totaux annuels
        cols_precip = [f"precip_{m:02d}_mm" for m in range(1,13)]
        row["precip_annuelle_mm"]        = sum(row.get(c, 0) for c in cols_precip)
        row["temp_moy_annuelle_c"]       = groupe["T2M"].mean()
        row["temp_max_annuelle_c"]       = groupe["T2M_MAX"].max()
        row["temp_min_annuelle_c"]       = groupe["T2M_MIN"].min()
        row["humidite_moy_annuelle_pct"] = groupe["RH2M"].mean()
        row["rayonnement_moyen_annuel"]  = groupe["ALLSKY_SFC_SW_DWN"].mean()

        etp_estimee = row["rayonnement_moyen_annuel"] * 0.408 * 30 * 12
        row["indice_stress_hydrique"] = (
            row["precip_annuelle_mm"] / etp_estimee
            if etp_estimee > 0 else None
        )

        resultats.append(row)

    return pd.DataFrame(resultats)

climat_annuel_communal = agreger_par_annee_commune(climat_mensuel_communal)
print(f"Shape : {climat_annuel_communal.shape}")
print(f"Vérification précipitations Abomey 2003 :")
print(climat_annuel_communal[
    climat_annuel_communal["commune"] == "Abomey"
][["annee","precip_annuelle_mm","temp_moy_annuelle_c"]].head(3))

Shape : (1540, 81)
Vérification précipitations Abomey 2003 :
   annee  precip_annuelle_mm  temp_moy_annuelle_c
0   2003             1321.57            26.264167
1   2004             1338.52            25.944167
2   2005             1068.37            26.005000


In [11]:
# Cellule 8 — type de sol par commune
pedologie = gpd.read_file("../data/raw/pedologie/Pédologie_SHP.shp")
communes_shp = gpd.read_file("../data/raw/limites/Lim_IGN_Communes.shp")
communes_shp = communes_shp.set_crs(epsg=32631)

def categoriser_sol(s):
    if not isinstance(s, str): return 'Inconnu'
    s = s.lower()
    if 'ferrallitique' in s:       return 'Ferrallitique'
    elif 'ferrugineux' in s:       return 'Ferrugineux tropical'
    elif 'hydromorphe' in s:       return 'Hydromorphe'
    elif 'vertisol' in s:          return 'Vertisol'
    elif 'brunifié' in s:          return 'Sol brun'
    elif 'peu évolué' in s:        return 'Sol peu évolué'
    elif 'minéral brut' in s:      return 'Sol minéral brut'
    else:                          return 'Autre'

pedologie['categorie_sol'] = pedologie['Types_sols'].apply(categoriser_sol)

# Intersection spatiale
intersection = gpd.overlay(
    communes_shp,
    pedologie[['categorie_sol', 'geometry']],
    how='intersection'
)
intersection['superficie'] = intersection.geometry.area

# Type de sol dominant par commune
sol_dominant = (
    intersection.groupby(['NAME', 'categorie_sol'])['superficie']
    .sum()
    .reset_index()
    .sort_values('superficie', ascending=False)
    .drop_duplicates('NAME')
    .rename(columns={'NAME': 'commune', 'categorie_sol': 'type_sol'})
    [['commune', 'type_sol']]
    .reset_index(drop=True)
)

print(f"Communes avec type de sol : {len(sol_dominant)}")
print(sol_dominant['type_sol'].value_counts())
print(sol_dominant.head(10))

Communes avec type de sol : 77
type_sol
Ferrugineux tropical    36
Ferrallitique           31
Hydromorphe              6
Vertisol                 3
Autre                    1
Name: count, dtype: int64
     commune              type_sol
0  Tchaourou  Ferrugineux tropical
1    Bassila  Ferrugineux tropical
2   Karimama  Ferrugineux tropical
3   Gogounou  Ferrugineux tropical
4    Segbana  Ferrugineux tropical
5  Tanguieta  Ferrugineux tropical
6  Banikoara  Ferrugineux tropical
7    Djougou  Ferrugineux tropical
8     N'Dali  Ferrugineux tropical
9      Kandi  Ferrugineux tropical


In [12]:
# Cellule 9 — fusion rendement + climat + type de sol
# 1. Harmoniser les noms de communes entre les trois sources
print("Noms dans rendement :", sorted(df_long["commune"].unique()))
print()
print("Noms dans climat    :", sorted(climat_annuel_communal["commune"].unique()))
print()
print("Noms dans sol       :", sorted(sol_dominant["commune"].unique()))

Noms dans rendement : ['Abomey', 'Abomey-Calavi', 'Adja-Ouèrè', 'Adjarra', 'Adjohoun', 'Agbangnizoun', 'Aguégués', 'Akpro-Missérété', 'Allada', 'Aplahoué', 'Athiémé', 'Avrankou', 'Banikoara', 'Bantè', 'Bassila', 'Bembèrèkè', 'Bohicon', 'Bonou', 'Bopa', 'Boukoumbé', 'Cobly', 'Comè', 'Copargo', 'Cotonou', 'Covè', 'Dangbo', 'Dassa-Zounmè', 'Djakotomey', 'Djidja', 'Djougou', 'Dogbo', 'Glazoué', 'Gogounou', 'Grand-Popo', 'Houéyogbé', 'Ifangni', 'Kalalé', 'Kandi', 'Karimama', 'Klouékanmè', 'Kouandé', 'Kpomassè', 'Kérou', 'Kétou', 'Lalo', 'Lokossa', 'Malanville', 'Matéri', "N'Dali", 'Natitingou', 'Nikki', 'Ouaké', 'Ouidah', 'Ouinhi', 'Ouèssè', 'Parakou', 'Pobè', 'Porto-Novo', 'Pèrèrè', 'Péhounco', 'Sakété', 'Savalou', 'Savè', 'Segbana', 'Sinendé', 'Sèmè-Podji', 'Sô-Ava', 'Tanguiéta', 'Tchaourou', 'Toffo', 'Tori-Bossito', 'Toucountouna', 'Toviklin', 'Za-Kpota', 'Zangnanado', 'Zogbodomey', 'Zè']

Noms dans climat    : ['Abomey', 'Abomey-Calavi', 'Adja-Ouere', 'Adjara', 'Adjohoun', 'Agbangnizoun

In [13]:
# Cellule 10 — harmonisation des noms
correspondance = {
    'Adja-Ouèrè':      'Adja-Ouere',
    'Adjarra':         'Adjara',
    'Aguégués':        'Aguegue',
    'Akpro-Missérété': 'Akpro-Misserete',
    'Aplahoué':        'Aplahoue',
    'Athiémé':         'Athieme',
    'Bantè':           'Bante',
    'Bembèrèkè':       'Bembereke',
    'Boukoumbé':       'Boukoumbe',
    'Cobly':           'Cobli',
    'Comè':            'Come',
    'Covè':            'Cove',
    'Dassa-Zounmè':    'Dassa-Zoume',
    'Djidja':          'Djidja',
    'Dogbo':           'Dogbo-Tota',
    'Glazoué':         'Glazoue',
    'Houéyogbé':       'Houeyogbe',
    'Kalalé':          'Kalale',
    'Klouékanmè':      'Klouekanme',
    'Kouandé':         'Kouande',
    'Kpomassè':        'Kpomasse',
    'Kérou':           'Kerou',
    'Kétou':           'Ketou',
    'Matéri':          'Materi',
    'Ouaké':           'Ouake',
    'Ouèssè':          'Ouesse',
    'Péhounco':        'Pehunco',
    'Pèrèrè':          'Perere',
    'Sakété':          'Sakete',
    'Savè':            'Save',
    'Sinendé':         'Sinende',
    'Sèmè-Podji':      'Seme-Kpodji',
    'Sô-Ava':          'So-Ava',
    'Tanguiéta':       'Tanguieta',
    'Toucountouna':    'Toukountouna',
    'Zangnanado':      'Zangnanado',
    'Zogbodomey':      'Zogbodome',
    'Zè':              'Ze',
}

df_long["commune"] = df_long["commune"].replace(correspondance)

# Vérifier ce qui ne matche pas encore
noms_rendement = set(df_long["commune"].unique())
noms_climat    = set(climat_annuel_communal["commune"].unique())

diff = noms_rendement - noms_climat
print(f"Communes sans correspondance : {diff}")

Communes sans correspondance : set()


In [14]:
# Cellule 11 — fusion des trois sources
dataset_communal = pd.merge(
    df_long,
    climat_annuel_communal,
    on=["commune", "annee"],
    how="inner"
)

dataset_communal = pd.merge(
    dataset_communal,
    sol_dominant,
    on="commune",
    how="left"
)

# Ajouter le département depuis les coordonnées
dataset_communal = pd.merge(
    dataset_communal,
    coords[["commune", "departement"]],
    on="commune",
    how="left"
)

# Nettoyer les noms de départements
dataset_communal["departement"] = dataset_communal["departement"].str.strip().str.title()

print(f"Shape final : {dataset_communal.shape}")
print(f"Communes    : {dataset_communal['commune'].nunique()}")
print(f"Départements: {dataset_communal['departement'].nunique()}")
print(f"Années      : {dataset_communal['annee'].min()} à {dataset_communal['annee'].max()}")
print(f"Valeurs manquantes : {dataset_communal.isnull().sum().sum()}")
print(dataset_communal.head(3))

Shape final : (1515, 85)
Communes    : 77
Départements: 12
Années      : 2003 à 2022
Valeurs manquantes : 0
     commune  annee  rendement_kg_ha  rendement_t_ha  precip_01_mm  \
0  Boukoumbe   2003      1019.387755        1.019388          7.75   
1      Cobli   2003      1622.168840        1.622169          1.55   
2      Kerou   2003      1770.690480        1.770690          0.31   

   temp_moy_01_c  temp_max_01_c  temp_min_01_c  humidite_01_pct  \
0          25.46          37.27          13.95            31.79   
1          25.46          37.66          13.12            29.41   
2          25.14          37.30          12.27            21.11   

   rayonnement_01  ...  rayonnement_12  precip_annuelle_mm  \
0           20.01  ...           20.09             1430.29   
1           19.72  ...           20.01             1423.82   
2           19.86  ...           20.14             1143.24   

   temp_moy_annuelle_c  temp_max_annuelle_c  temp_min_annuelle_c  \
0            26.323333   

In [15]:
# Cellule 12 — feature engineering
# Précipitations saisonnières
cols_s1 = [f"precip_{m:02d}_mm" for m in [3,4,5,6,7]]
cols_s2 = [f"precip_{m:02d}_mm" for m in [9,10,11]]
cols_croissance = [f"precip_{m:02d}_mm" for m in [5,6,7,8,9,10]]

dataset_communal["precip_saison1_mm"]    = dataset_communal[cols_s1].sum(axis=1)
dataset_communal["precip_saison2_mm"]    = dataset_communal[cols_s2].sum(axis=1)
dataset_communal["precip_croissance_mm"] = dataset_communal[cols_croissance].sum(axis=1)

# Amplitude thermique mensuelle
for m in range(1, 13):
    dataset_communal[f"amplitude_{m:02d}_c"] = (
        dataset_communal[f"temp_max_{m:02d}_c"] - dataset_communal[f"temp_min_{m:02d}_c"]
    )
dataset_communal["amplitude_annuelle_c"] = (
    pd.DataFrame([dataset_communal[f"amplitude_{m:02d}_c"] for m in range(1,13)]).T.mean(axis=1)
)

# Stress thermique
for m in range(1, 13):
    dataset_communal[f"stress_thermique_{m:02d}"] = (
        dataset_communal[f"temp_max_{m:02d}_c"] > 35
    ).astype(int)
cols_stress = [f"stress_thermique_{m:02d}" for m in range(1,13)]
dataset_communal["nb_mois_stress_thermique"] = dataset_communal[cols_stress].sum(axis=1)

# Variable temporelle
dataset_communal["annee_normalisee"] = (
    dataset_communal["annee"] - 2003
) / (2022 - 2003)

# Encodage département et type de sol
from sklearn.preprocessing import LabelEncoder
le_dept = LabelEncoder()
le_sol  = LabelEncoder()
dataset_communal["departement_encode"] = le_dept.fit_transform(dataset_communal["departement"])
dataset_communal["type_sol_encode"]    = le_sol.fit_transform(dataset_communal["type_sol"])

print("Encodage départements:", dict(zip(le_dept.classes_, le_dept.transform(le_dept.classes_))))
print("Encodage types de sol:", dict(zip(le_sol.classes_, le_sol.transform(le_sol.classes_))))

# Sauvegarde
dataset_communal = dataset_communal.copy()
dataset_communal.to_csv("../data/processed/dataset_communal.csv", index=False)

print(f"\nShape final enrichi : {dataset_communal.shape}")
print(f"Sauvegardé dans data/processed/dataset_communal.csv")

Encodage départements: {'Alibori': np.int64(0), 'Atacora': np.int64(1), 'Atlantique': np.int64(2), 'Borgou': np.int64(3), 'Collines': np.int64(4), 'Couffo': np.int64(5), 'Donga': np.int64(6), 'Littoral': np.int64(7), 'Mono': np.int64(8), 'Oueme': np.int64(9), 'Plateau': np.int64(10), 'Zou': np.int64(11)}
Encodage types de sol: {'Autre': np.int64(0), 'Ferrallitique': np.int64(1), 'Ferrugineux tropical': np.int64(2), 'Hydromorphe': np.int64(3), 'Vertisol': np.int64(4)}

Shape final enrichi : (1515, 117)
Sauvegardé dans data/processed/dataset_communal.csv
